In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install Biopython --q
!pip install goatools --q
!pip install iterative-stratification --q

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from Bio import SeqIO

In [ ]:
# Load protein sequences
from Bio import SeqIO
import pandas as pd
train_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta"
test_path = "/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta"

train_records = list(SeqIO.parse(train_path,"fasta"))
df_train = pd.DataFrame({
    "protein_id":[rec.id.split("|")[1] for rec in train_records],
    "sequence":[str(rec.seq) for rec in train_records],
})

test_records = list(SeqIO.parse(test_path,"fasta"))
df_test = pd.DataFrame({
    "protein_id":[rec.id.split(" ")[0] for rec in test_records],
    "sequence":[str(rec.seq) for rec in test_records],
})

# Load train_taxonomy
taxonomy_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv"
df_tax = pd.read_csv(taxonomy_path,sep="\t", header=None)
df_tax.columns = ["protein_id", "taxon_id"]

train_terms_path = "/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv"

df_terms = pd.read_csv(train_terms_path, sep="\t")

df_terms.columns = ["protein_id", "go_id", "aspect"]
df_terms.head()

# Combine everything into a unified table
df_train = df_train.merge(df_tax, on = "protein_id",how="left")

terms_per_protein = (
    df_terms.groupby("protein_id")["go_id"]
    .apply(list)
    .to_dict()
)

df_train["go_terms"] = df_train["protein_id"].map(terms_per_protein).fillna("").apply(
    lambda x:x if isinstance(x,list) else[]
)

print("Proteins:", len(df_train))
print("Proteins with GO terms:", df_train['go_terms'].apply(len).gt(0).sum())
print("Unique GO terms:", df_terms['go_id'].nunique())
print("Aspect distribution:", df_terms['aspect'].value_counts())


In [ ]:
df_train.head()

In [ ]:
# Load test dataset
test_records = list(SeqIO.parse(test_path, "fasta"))

protein_ids = []
taxon_ids = []
sequences = []

for rec in test_records:
    header = rec.description.split()
    protein_id = header[0]          # A0A0C5B5G6
    taxon_id = header[1]            # 9606
    
    protein_ids.append(protein_id)
    taxon_ids.append(taxon_id)
    sequences.append(str(rec.seq))

df_test = pd.DataFrame({
    "protein_id": protein_ids,
    "taxon_id": taxon_ids,
    "sequence": sequences
})

df_tax_species=pd.read_csv("/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset-taxon-list.tsv", sep="\t")
df_tax_species.columns=["taxon_id","species"]
df_tax_species["taxon_id"] = df_tax_species["taxon_id"].astype(str)
df_test = df_test.merge(df_tax_species, on="taxon_id", how="left")
df_test.head()

In [ ]:
# GO label propagation

from goatools.obo_parser import GODag
go_dag = GODag("/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo")

def get_ancestors(go_id,go_dag):
    if go_id not in go_dag:
        return set()
    term = go_dag[go_id]
    ancestors = term.get_all_parents()
    ancestors.add(go_id)
    return ancestors

# Propagate GO terms for one protein

def propagate_terms(go_list, go_dag):
    """Propagate a list of GO terms to include all ancestors."""
    propagated = set()
    for go_id in go_list:
        if go_id not in go_dag:
            continue
        term = go_dag[go_id]
        propagated.add(go_id)
        ancestors = term.get_all_parents()
        propagated |= ancestors
    return propagated


df_train["go_terms_propagated"]=df_train["go_terms"].apply(
    lambda terms: propagate_terms(terms,go_dag)
)

# Split MF / BP / C
def split_by_namespace(go_set, go_dag):
    mf, bp, cc = [], [], []
    for go_id in go_set:
        if go_id in go_dag:
            ns = go_dag[go_id].namespace
            if ns == "molecular_function":
                mf.append(go_id)
            elif ns == "biological_process":
                bp.append(go_id)
            elif ns == "cellular_component":
                cc.append(go_id)
    return mf, bp, cc

df_train["MF"], df_train["BP"], df_train["CC"] = zip(*df_train["go_terms_propagated"].apply(
    lambda s: split_by_namespace(s, go_dag)
))

df_train.head()



In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb_mf = MultiLabelBinarizer()
Y_mf = mlb_mf.fit_transform(df_train["MF"])

mlb_bp = MultiLabelBinarizer()
Y_bp = mlb_bp.fit_transform(df_train["BP"])

mlb_cc = MultiLabelBinarizer()
Y_cc = mlb_cc.fit_transform(df_train["CC"])


print("Avg MF labels:", Y_mf.sum(axis=1).mean())
print("Avg BP labels:", Y_bp.sum(axis=1).mean())
print("Avg CC labels:", Y_cc.sum(axis=1).mean())

print("Y_mf.shape:",Y_mf.shape)
print("Y_bp.shape:",Y_bp.shape)
print("Y_cc.shape:",Y_cc.shape)

In [ ]:
import joblib


joblib.dump(mlb_mf, "/kaggle/working/mlb_mf.pkl")
joblib.dump(mlb_bp, "/kaggle/working/mlb_bp.pkl")
joblib.dump(mlb_cc, "/kaggle/working/mlb_cc.pkl")

In [ ]:
# filter rare GO terms

ROOT_TERMS = {'GO:0008150', 'GO:0003674', 'GO:0005575'}

min_count = 30  # common threshold

mf_counts = Y_mf.sum(axis=0)
keep_mf = [
    (count >= min_count) and (term not in ROOT_TERMS)
    for count, term in zip(mf_counts, mlb_mf.classes_)
]
Y_mf = Y_mf[:, keep_mf]
mf_terms = [t for t, k in zip(mlb_mf.classes_, keep_mf) if k]


min_count = 50  # common threshold
bp_counts = Y_bp.sum(axis=0)
keep_bp = [
    (count >= min_count) and (term not in ROOT_TERMS)
    for count, term in zip(bp_counts, mlb_bp.classes_)
]
Y_bp = Y_bp[:, keep_bp]
bp_terms = [t for t, k in zip(mlb_bp.classes_, keep_bp) if k]

min_count = 20
cc_counts = Y_cc.sum(axis=0)
keep_cc = [
    (count >= min_count) and (term not in ROOT_TERMS)
    for count, term in zip(cc_counts, mlb_cc.classes_)
]
Y_cc = Y_cc[:, keep_cc]
cc_terms = [t for t, k in zip(mlb_cc.classes_, keep_cc) if k]



print("Y_mf.shape:",Y_mf.shape)
print("Y_bp.shape:",Y_bp.shape)
print("Y_cc.shape:",Y_cc.shape)

np.save("Y_mf.npy", Y_mf)
np.save("Y_bp.npy", Y_bp)
np.save("Y_cc.npy", Y_cc)




In [ ]:

print(f"Total proteins: {len(df_train):,}")
print(f"MF terms: {Y_mf.shape[1]}")
print(f"BP terms: {Y_bp.shape[1]}")
print(f"CC terms: {Y_cc.shape[1]}")
print(f"Total GO terms: {Y_mf.shape[1] + Y_bp.shape[1] + Y_cc.shape[1]}")

print(f"MF: avg {Y_mf.sum(axis=1).mean():.1f} terms/protein, "
      f"avg {Y_mf.sum(axis=0).mean():.1f} proteins/term")
print(f"BP: avg {Y_bp.sum(axis=1).mean():.1f} terms/protein, "
      f"avg {Y_bp.sum(axis=0).mean():.1f} proteins/term")
print(f"CC: avg {Y_cc.sum(axis=1).mean():.1f} terms/protein, "
      f"avg {Y_cc.sum(axis=0).mean():.1f} proteins/term")

print(f"MF: {Y_mf.mean()*100:.2f}% positive labels")
print(f"BP: {Y_bp.mean()*100:.2f}% positive labels")
print(f"CC: {Y_cc.mean()*100:.2f}% positive labels")
# Verify no root terms remain
assert not any(term in ROOT_TERMS for term in mf_terms), "MF still has root!"
assert not any(term in ROOT_TERMS for term in bp_terms), "BP still has root!"
assert not any(term in ROOT_TERMS for term in cc_terms), "CC still has root!"

print("No root terms in training labels")

In [ ]:
# Load the protT5 embeddings
train_path = "/kaggle/input/train-embeddings-prott5-xl-half/train_prott5_xl_half_embeddings.npy"
test_path = "/kaggle/input/prott5-embeddings-test-npy/test_prott5_embeddings.npy"

X_np = np.load(train_path)
X_train_torch = torch.from_numpy(X_np).float()

test  =  np.load(test_path)
X_test_torch = torch.from_numpy(test).float()

def load_ids(fasta):
    return [r.id.split()[0] for r in SeqIO.parse(fasta,"fasta")]

train_ids = load_ids("/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta")
print(train_ids[0])
train_ids = [id.split("|")[1] for id in train_ids]
test_ids = load_ids("/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta")

print(train_ids[0],len(train_ids))
print(test_ids[0],len(test_ids))

In [ ]:
train_embedding_dict = {
    pid: X_np[i]
    for i, pid in enumerate(train_ids)
}
test_embedding_dict = {
    pid: test[i]
    for i, pid in enumerate(test_ids)
}

X_train = np.vstack([train_embedding_dict[pid] for pid in df_train["protein_id"]])
assert len(df_train) == len(X_train), "Length mismatch!"

missing = set(df_train["protein_id"]) - set(train_embedding_dict.keys())
assert len(missing) == 0, f"Missing embeddings: {len(missing)}"



X_test = np.vstack([test_embedding_dict[pid] for pid in df_test["protein_id"]])
np.save("X_train_protT5.npy", X_train)
np.save("X_test_protT5.npy", X_test)

In [ ]:
go_counts = (
    np.asarray(Y_mf.sum(axis=1)).ravel() +
    np.asarray(Y_bp.sum(axis=1)).ravel() +
    np.asarray(Y_cc.sum(axis=1)).ravel()
)

go_counts_capped = np.clip(go_counts, 0, 20)

bins = pd.qcut(
    go_counts_capped,
    q=10,
    duplicates="drop"
)

bins.value_counts().sort_index()

In [ ]:
from sklearn.model_selection import train_test_split
# Split

idx = np.arange(len(go_counts))

train_idx, val_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42,
    stratify=bins
)

X_tr = X_train[train_idx]
X_val = X_train[val_idx]

Y_mf_tr, Y_mf_val = Y_mf[train_idx], Y_mf[val_idx]
Y_bp_tr, Y_bp_val = Y_bp[train_idx], Y_bp[val_idx]
Y_cc_tr, Y_cc_val = Y_cc[train_idx], Y_cc[val_idx]

In [ ]:
np.save("X_tr", X_tr)
np.save("X_val", X_val)

np.save("Y_mf_tr", Y_mf_tr)
np.save("Y_mf_val",Y_mf_val)
np.save("Y_bp_tr", Y_bp_tr)
np.save("Y_bp_val",Y_bp_val)
np.save("Y_cc_tr", Y_cc_tr)
np.save("Y_cc_val",Y_cc_val)

In [ ]:
print("Train bin distribution:")
print(pd.qcut(go_counts[train_idx], q=10, duplicates="drop").value_counts().sort_index())

print("\nVal bin distribution:")
print(pd.qcut(go_counts[val_idx], q=10, duplicates="drop").value_counts().sort_index())

In [ ]:
class MF_MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 2048),
            nn.LayerNorm(2048),
            nn.ReLU(),
            nn.Dropout(0.4),
            
                      
            nn.Linear(2048, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class BP_MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(in_dim, 2048),  # Increased from 512
            nn.LayerNorm(2048),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(2048, 1024),  # Increased from 256
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(1024, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class CC_MLP(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(512, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)


In [ ]:

# Train the model

def train_aspect(
    model_class, 
    X_train, Y_train, 
    X_val, Y_val, 
    aspect_name="",
    pos_weight_alpha=0.5,  # Smoothing factor
    epochs=50, 
    batch_size=4096, 
    lr=1e-3, 
    patience=10
):
    """
    Train a single-aspect model.
    
    Args:
        model_class: Model class (MF_MLP, BP_MLP, or CC_MLP)
        X_train, Y_train: Training data (numpy arrays or tensors)
        X_val, Y_val: Validation data
        aspect_name: Name for logging (MF/BP/CC)
        pos_weight_alpha: Smoothing for pos_weight (0.5 = moderate)
        epochs: Maximum epochs
        batch_size: Batch size
        lr: Learning rate
        patience: Early stopping patience
    
    Returns:
        Trained model
    """
    print(f"\n{'='*60}")
    print(f"Training {aspect_name} Model")
    print(f"{'='*60}")
    
    # Convert to tensors if needed
    if not isinstance(X_train, torch.Tensor):
        X_train = torch.tensor(X_train, dtype=torch.float32)
    if not isinstance(Y_train, torch.Tensor):
        Y_train = torch.tensor(Y_train, dtype=torch.float32)
    if not isinstance(X_val, torch.Tensor):
        X_val = torch.tensor(X_val, dtype=torch.float32)
    if not isinstance(Y_val, torch.Tensor):
        Y_val = torch.tensor(Y_val, dtype=torch.float32)
    
    # Move validation data to GPU once (make copies to avoid modifying originals)
    X_val_gpu = X_val.to(device)
    Y_val_gpu = Y_val.to(device)
    
    # Compute pos_weight with smoothing
    pos_freq = Y_train.mean(dim=0)  # Frequency of each class
    pos_freq = torch.clamp(pos_freq, min=0.001, max=0.999)  # Avoid division by zero
    
    # Smooth pos_weight: ((1-p)/p)^alpha
    pos_weight = ((1 - pos_freq) / pos_freq) ** pos_weight_alpha
    pos_weight = pos_weight.to(device)
    
    print(f"Class balance: {pos_freq.mean().item()*100:.2f}% positive")
    print(f"Avg pos_weight: {pos_weight.mean().item():.1f}x")
    print(f"Training samples: {len(X_train):,}")
    print(f"Output dimensions: {Y_train.shape[1]}")
    
    # DataLoader
    dataset = TensorDataset(X_train, Y_train)
    loader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        pin_memory=True,
        num_workers=0 
    )
    
    # Initialize model
    model = model_class(X_train.shape[1], Y_train.shape[1]).to(device)
    
    # Optimizer & loss
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )
    
    # Early stopping
    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None
    
    print(f"\nStarting training...")
    
    for epoch in range(epochs):
    
        model.train()
        train_loss = 0.0
        
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
        
        train_loss /= len(X_train)
        
        # ==================
        # VALIDATION
        # ==================
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_gpu)
            val_loss = loss_fn(val_logits, Y_val_gpu).item()
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Print progress
        print(f"Epoch {epoch+1:3d}/{epochs}: "
              f"train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}, "
              f"lr={optimizer.param_groups[0]['lr']:.6f}")
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict()
            print(f"  → New best validation loss: {val_loss:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\nEarly stopping triggered after {epoch+1} epochs")
                break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✓ Best model restored (val_loss={best_val_loss:.4f})")
    
    return model





In [ ]:
def compute_fmax(preds, labels, thresholds=None):
    """
    CAFA-style protein-centric Fmax.
    
    Args:
        preds: (n_proteins, n_terms) prediction scores
        labels: (n_proteins, n_terms) binary labels
        thresholds: array of thresholds to try
    
    Returns:
        best_fmax, best_threshold
    """
    if thresholds is None:
        # Use finer thresholds in low range where optimal often is
        thresholds = np.concatenate([
            np.arange(0.01, 0.1, 0.01),   # Fine-grained for low thresholds
            np.arange(0.1, 0.5, 0.02),    # Medium-grained
            np.arange(0.5, 1.0, 0.05)     # Coarse for high thresholds
        ])
    
    best_f = 0
    best_t = 0
    n_proteins = labels.shape[0]
    
    for t in thresholds:
        # Binarize predictions at threshold t
        preds_binary = (preds >= t).astype(np.int32)
        
        # Vectorized computation per protein
        tp = np.sum(labels & preds_binary, axis=1)  # True positives per protein
        fp = np.sum((1 - labels) & preds_binary, axis=1)  # False positives per protein
        fn = np.sum(labels & (1 - preds_binary), axis=1)  # False negatives per protein
        
        # Recall for all proteins (can be 0)
        recall_denom = tp + fn
        recalls = np.zeros(n_proteins)
        valid_recall = recall_denom > 0
        recalls[valid_recall] = tp[valid_recall] / recall_denom[valid_recall]
        
        # Precision only for proteins with predictions
        precision_denom = tp + fp
        has_predictions = precision_denom > 0
        
        if not np.any(has_predictions):
            # No predictions at this threshold, skip
            continue
        
        precisions = tp[has_predictions] / precision_denom[has_predictions]
        
        # Average precision (only over proteins with predictions)
        # Average recall (over ALL proteins)
        P = np.mean(precisions)
        R = np.mean(recalls)
        
        if P + R > 0:
            F = 2 * P * R / (P + R)
        else:
            F = 0
        
        if F > best_f:
            best_f = F
            best_t = t
    
    return best_f, best_t

In [ ]:
def compute_fmax_with_details(preds, labels, thresholds=None):
    """
    Extended version that returns precision and recall at optimal threshold.
    """
    if thresholds is None:
        thresholds = np.concatenate([
            np.arange(0.01, 0.1, 0.01),
            np.arange(0.1, 0.5, 0.02),
            np.arange(0.5, 1.0, 0.05)
        ])
    
    best_f = 0
    best_t = 0
    best_p = 0
    best_r = 0
    n_proteins = labels.shape[0]
    
    for t in thresholds:
        preds_binary = (preds >= t).astype(np.int32)
        
        tp = np.sum(labels & preds_binary, axis=1)
        fp = np.sum((1 - labels) & preds_binary, axis=1)
        fn = np.sum(labels & (1 - preds_binary), axis=1)
        
        recall_denom = tp + fn
        recalls = np.zeros(n_proteins)
        valid_recall = recall_denom > 0
        recalls[valid_recall] = tp[valid_recall] / recall_denom[valid_recall]
        
        precision_denom = tp + fp
        has_predictions = precision_denom > 0
        
        if not np.any(has_predictions):
            continue
        
        precisions = tp[has_predictions] / precision_denom[has_predictions]
        
        P = np.mean(precisions)
        R = np.mean(recalls)
        
        if P + R > 0:
            F = 2 * P * R / (P + R)
        else:
            F = 0
        
        if F > best_f:
            best_f = F
            best_t = t
            best_p = P
            best_r = R
    
    return best_f, best_t, best_p, best_r

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


# Convert validation data to tensors once (for evaluation later)
if not isinstance(X_val, torch.Tensor):
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
else:
    X_val_tensor = X_val
    
# Train MF model
model_mf = train_aspect(
    MF_MLP, 
    X_tr, Y_mf_tr, 
    X_val, Y_mf_val,
    aspect_name="MF",
    pos_weight_alpha=0,
    epochs=50,
    batch_size=1024,
    lr=1e-3,
    patience=10
)
    
# Evaluate MF
model_mf.eval()
with torch.no_grad():
    pred_mf = torch.sigmoid(model_mf(X_val_tensor.to(device))).cpu().numpy()
    
fmax_mf, thresh_mf, prec_mf, rec_mf = compute_fmax_with_details(pred_mf, Y_mf_val if isinstance(Y_mf_val, np.ndarray) else Y_mf_val.numpy())
print(f"\n{'='*60}")
print(f"MF Results: Fmax={fmax_mf:.4f} @ t={thresh_mf:.3f} "
        f"(P={prec_mf:.3f}, R={rec_mf:.3f})")
print(f"{'='*60}")
    
# Train BP model
model_bp = train_aspect(
    BP_MLP, 
    X_tr, Y_bp_tr, 
    X_val, Y_bp_val,
    aspect_name="BP",
    pos_weight_alpha=0,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    patience=15
)
    
# Evaluate BP
model_bp.eval()
with torch.no_grad():
    pred_bp = torch.sigmoid(model_bp(X_val_tensor.to(device))).cpu().numpy()
    
fmax_bp, thresh_bp, prec_bp, rec_bp = compute_fmax_with_details(pred_bp, Y_bp_val if isinstance(Y_bp_val, np.ndarray) else Y_bp_val.numpy())
print(f"\n{'='*60}")
print(f"BP Results: Fmax={fmax_bp:.4f} @ t={thresh_bp:.3f} "
        f"(P={prec_bp:.3f}, R={rec_bp:.3f})")
print(f"{'='*60}")
    
# Train CC model
model_cc = train_aspect(
    CC_MLP, 
    X_tr, Y_cc_tr, 
    X_val, Y_cc_val,
    aspect_name="CC",
    pos_weight_alpha=0,
    epochs=50,
    batch_size=1024,
    lr=1e-3,
    patience=10
)
    
# Evaluate CC
model_cc.eval()
with torch.no_grad():
    pred_cc = torch.sigmoid(model_cc(X_val_tensor.to(device))).cpu().numpy()
    
fmax_cc, thresh_cc, prec_cc, rec_cc = compute_fmax_with_details(pred_cc, Y_cc_val if isinstance(Y_cc_val, np.ndarray) else Y_cc_val.numpy())
print(f"\n{'='*60}")
print(f"CC Results: Fmax={fmax_cc:.4f} @ t={thresh_cc:.3f} "
        f"(P={prec_cc:.3f}, R={rec_cc:.3f})")
print(f"{'='*60}")
    
# Overall results
avg_fmax = (fmax_mf + fmax_bp + fmax_cc) / 3
print(f"\n{'='*60}")
print(f"OVERALL RESULTS")
print(f"{'='*60}")
print(f"Average Fmax: {avg_fmax:.4f}")
print(f"MF: {fmax_mf:.4f}")
print(f"BP: {fmax_bp:.4f}")
print(f"CC: {fmax_cc:.4f}")
    
# Save models
torch.save(model_mf.state_dict(), "model_mf_best.pt")
torch.save(model_bp.state_dict(), "model_bp_best.pt")
torch.save(model_cc.state_dict(), "model_cc_best.pt")
print(f"\n✓ Models saved!")
    
    

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import pickle

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ============================================
# LOAD TRAINED MODELS
# ============================================

print("\n" + "="*60)
print("LOADING TRAINED MODELS")
print("="*60)

# Initialize models
mf_model = MF_MLP(X_val.shape[1], Y_mf_val.shape[1]).to(device)
bp_model = BP_MLP(X_val.shape[1], Y_bp_val.shape[1]).to(device)
cc_model = CC_MLP(X_val.shape[1], Y_cc_val.shape[1]).to(device)

# Load state dicts
try:
    mf_model.load_state_dict(torch.load("/kaggle/working/model_mf_best.pt", map_location=device))
    print("✓ MF model loaded")
except FileNotFoundError:
    print("✗ MF model not found!")

try:
    bp_model.load_state_dict(torch.load("/kaggle/working/model_bp_best.pt", map_location=device))
    print("✓ BP model loaded")
except FileNotFoundError:
    print("✗ BP model not found!")

try:
    cc_model.load_state_dict(torch.load("/kaggle/working/model_cc_best.pt", map_location=device))
    print("✓ CC model loaded")
except FileNotFoundError:
    print("✗ CC model not found!")

# Set to evaluation mode
mf_model.eval()
bp_model.eval()
cc_model.eval()

print("✓ All models set to eval mode")

# ============================================
# VALIDATION SET PREDICTIONS
# ============================================

print("\n" + "="*60)
print("GENERATING VALIDATION PREDICTIONS")
print("="*60)

# Convert to tensor
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)

# Generate predictions
with torch.no_grad():
    print("Predicting MF...")
    mf_val_logits = mf_model(X_val_tensor)
    mf_val_pred = torch.sigmoid(mf_val_logits).cpu().numpy()
    
    print("Predicting BP...")
    bp_val_logits = bp_model(X_val_tensor)
    bp_val_pred = torch.sigmoid(bp_val_logits).cpu().numpy()
    
    print("Predicting CC...")
    cc_val_logits = cc_model(X_val_tensor)
    cc_val_pred = torch.sigmoid(cc_val_logits).cpu().numpy()

print("\n✓ Predictions generated")

# Verify shapes
print("\nShape verification:")
print(f"  MF: {mf_val_pred.shape} (expected: {Y_mf_val.shape})")
print(f"  BP: {bp_val_pred.shape} (expected: {Y_bp_val.shape})")
print(f"  CC: {cc_val_pred.shape} (expected: {Y_cc_val.shape})")

assert mf_val_pred.shape == (Y_mf_val.shape if isinstance(Y_mf_val, np.ndarray) else Y_mf_val.numpy().shape), "MF shape mismatch!"
assert bp_val_pred.shape == (Y_bp_val.shape if isinstance(Y_bp_val, np.ndarray) else Y_bp_val.numpy().shape), "BP shape mismatch!"
assert cc_val_pred.shape == (Y_cc_val.shape if isinstance(Y_cc_val, np.ndarray) else Y_cc_val.numpy().shape), "CC shape mismatch!"

# Save predictions
np.save("mlp_val_mf.npy", mf_val_pred)
np.save("mlp_val_bp.npy", bp_val_pred)
np.save("mlp_val_cc.npy", cc_val_pred)

print("\n✓ Predictions saved to .npy files")

# ============================================
# PREDICTION STATISTICS
# ============================================

print("\n" + "="*60)
print("PREDICTION STATISTICS")
print("="*60)

print("\nMF Predictions:")
print(f"  Mean: {mf_val_pred.mean():.4f}")
print(f"  Median: {np.median(mf_val_pred):.4f}")
print(f"  Std: {mf_val_pred.std():.4f}")
print(f"  Min: {mf_val_pred.min():.4f}, Max: {mf_val_pred.max():.4f}")
print(f"  % > 0.5: {(mf_val_pred > 0.5).mean()*100:.2f}%")

print("\nBP Predictions:")
print(f"  Mean: {bp_val_pred.mean():.4f}")
print(f"  Median: {np.median(bp_val_pred):.4f}")
print(f"  Std: {bp_val_pred.std():.4f}")
print(f"  Min: {bp_val_pred.min():.4f}, Max: {bp_val_pred.max():.4f}")
print(f"  % > 0.5: {(bp_val_pred > 0.5).mean()*100:.2f}%")

print("\nCC Predictions:")
print(f"  Mean: {cc_val_pred.mean():.4f}")
print(f"  Median: {np.median(cc_val_pred):.4f}")
print(f"  Std: {cc_val_pred.std():.4f}")
print(f"  Min: {cc_val_pred.min():.4f}, Max: {cc_val_pred.max():.4f}")
print(f"  % > 0.5: {(cc_val_pred > 0.5).mean()*100:.2f}%")

# ============================================
# COMPUTE FMAX
# ============================================

print("\n" + "="*60)
print("COMPUTING FMAX SCORES")
print("="*60)

# Convert labels to numpy if needed
Y_mf_val_np = Y_mf_val if isinstance(Y_mf_val, np.ndarray) else Y_mf_val.numpy()
Y_bp_val_np = Y_bp_val if isinstance(Y_bp_val, np.ndarray) else Y_bp_val.numpy()
Y_cc_val_np = Y_cc_val if isinstance(Y_cc_val, np.ndarray) else Y_cc_val.numpy()

# Compute Fmax for each aspect
print("\nComputing MF Fmax...")
fmax_mf, thresh_mf, prec_mf, rec_mf = compute_fmax_with_details(mf_val_pred, Y_mf_val_np)

print("Computing BP Fmax...")
fmax_bp, thresh_bp, prec_bp, rec_bp = compute_fmax_with_details(bp_val_pred, Y_bp_val_np)

print("Computing CC Fmax...")
fmax_cc, thresh_cc, prec_cc, rec_cc = compute_fmax_with_details(cc_val_pred, Y_cc_val_np)

# ============================================
# FINAL RESULTS
# ============================================

print("\n" + "="*60)
print("FINAL VALIDATION RESULTS")
print("="*60)

print(f"\nMF:")
print(f"  Fmax:      {fmax_mf:.4f}")
print(f"  Threshold: {thresh_mf:.3f}")
print(f"  Precision: {prec_mf:.3f}")
print(f"  Recall:    {rec_mf:.3f}")

print(f"\nBP:")
print(f"  Fmax:      {fmax_bp:.4f}")
print(f"  Threshold: {thresh_bp:.3f}")
print(f"  Precision: {prec_bp:.3f}")
print(f"  Recall:    {rec_bp:.3f}")

print(f"\nCC:")
print(f"  Fmax:      {fmax_cc:.4f}")
print(f"  Threshold: {thresh_cc:.3f}")
print(f"  Precision: {prec_cc:.3f}")
print(f"  Recall:    {rec_cc:.3f}")

avg_fmax = (fmax_mf + fmax_bp + fmax_cc) / 3

print(f"\n" + "="*60)
print(f"AVERAGE FMAX: {avg_fmax:.4f}")
print(f"="*60)

# Save thresholds for test set inference
optimal_thresholds = {
    'MF': thresh_mf,
    'BP': thresh_bp,
    'CC': thresh_cc
}

import pickle
with open('optimal_thresholds.pkl', 'wb') as f:
    pickle.dump(optimal_thresholds, f)

print("\n✓ Optimal thresholds saved to 'optimal_thresholds.pkl'")


In [ ]:
print("MF:", mf_val_pred.min(), mf_val_pred.max())
print("BP:", bp_val_pred.min(), bp_val_pred.max())
print("CC:", cc_val_pred.min(), cc_val_pred.max())

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

In [ ]:
# ============================================
# STEP 1: TRAIN MULTIPLE MODELS WITH DIFFERENT SEEDS
# ============================================

def set_seed(seed):
    """Set random seed for reproducibility"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def train_ensemble_models(
    model_class,
    X_train, Y_train,
    X_val, Y_val,
    aspect_name="",
    n_models=5,
    seeds=None,
    **training_kwargs  # All other train_aspect parameters
):
    """
    Train multiple models with different random seeds.
    
    Args:
        model_class: Model class (MF_MLP, BP_MLP, CC_MLP)
        X_train, Y_train: Training data
        X_val, Y_val: Validation data
        aspect_name: Name for logging
        n_models: Number of models to train
        seeds: List of seeds (if None, use [0, 1, 2, 3, 4])
        **training_kwargs: All parameters for train_aspect()
    
    Returns:
        List of trained models
    """
    if seeds is None:
        seeds = list(range(n_models))
    
    print(f"\n{'='*60}")
    print(f"TRAINING ENSEMBLE: {n_models} {aspect_name} MODELS")
    print(f"{'='*60}")
    
    models = []
    
    for i, seed in enumerate(seeds):
        print(f"\n--- Model {i+1}/{n_models} (seed={seed}) ---")
        
        # Set seed
        set_seed(seed)
        
        # Train model
        model = train_aspect(
            model_class,
            X_train, Y_train,
            X_val, Y_val,
            aspect_name=f"{aspect_name}_seed{seed}",
            **training_kwargs
        )
        
        models.append(model)
        
        # Save individual model
        torch.save(model.state_dict(), f"model_{aspect_name.lower()}_seed{seed}.pt")
        print(f"✓ Model saved: model_{aspect_name.lower()}_seed{seed}.pt")
    
    print(f"\n✓ Ensemble training complete: {n_models} models")
    return models


In [ ]:
# ============================================
# STEP 2: ENSEMBLE PREDICTION STRATEGIES
# ============================================

def ensemble_predict_average(models, X, device='cuda'):
    """
    Simple averaging: Average predictions from all models.
    
    This is the most common and often best-performing ensemble method.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    
    predictions = []
    
    for model in models:
        model.eval()
        with torch.no_grad():
            logits = model(X_tensor)
            preds = torch.sigmoid(logits).cpu().numpy()
            predictions.append(preds)
    
    # Simple average
    ensemble_pred = np.mean(predictions, axis=0)
    
    return ensemble_pred

In [ ]:
def ensemble_predict_weighted(models, X, weights, device='cuda'):
    """
    Weighted averaging: Weight models by their validation performance.
    
    Args:
        models: List of models
        X: Input data
        weights: List of weights (should sum to 1.0)
        device: Device to use
    
    Returns:
        Weighted ensemble predictions
    """
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    
    predictions = []
    
    for model in models:
        model.eval()
        with torch.no_grad():
            logits = model(X_tensor)
            preds = torch.sigmoid(logits).cpu().numpy()
            predictions.append(preds)
    
    # Weighted average
    ensemble_pred = np.zeros_like(predictions[0])
    for pred, weight in zip(predictions, weights):
        ensemble_pred += pred * weight
    
    return ensemble_pred


In [ ]:
def ensemble_predict_rank_average(models, X, device='cuda'):
    """
    Rank averaging: Convert predictions to ranks, then average.
    
    More robust to different prediction scales across models.
    """
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    
    predictions = []
    
    for model in models:
        model.eval()
        with torch.no_grad():
            logits = model(X_tensor)
            preds = torch.sigmoid(logits).cpu().numpy()
            predictions.append(preds)
    
    # Convert to ranks (per protein)
    ranked_predictions = []
    for pred in predictions:
        ranked = np.zeros_like(pred)
        for i in range(pred.shape[0]):
            # Rank predictions for this protein
            ranks = pred[i].argsort().argsort()
            # Normalize to [0, 1]
            ranked[i] = ranks / (pred.shape[1] - 1)
        ranked_predictions.append(ranked)
    
    # Average ranks
    ensemble_pred = np.mean(ranked_predictions, axis=0)
    
    return ensemble_pred


In [ ]:

# ============================================
# STEP 3: COMPUTE ENSEMBLE WEIGHTS FROM VALIDATION
# ============================================

def compute_ensemble_weights(models, X_val, Y_val, device='cuda'):
    """
    Compute optimal weights based on validation Fmax.
    
    Args:
        models: List of models
        X_val: Validation input
        Y_val: Validation labels
        device: Device
    
    Returns:
        List of weights (sum to 1.0)
    """
    print("\nComputing ensemble weights from validation performance...")
    
    fmax_scores = []
    
    for i, model in enumerate(models):
        model.eval()
        X_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
        
        with torch.no_grad():
            logits = model(X_tensor)
            preds = torch.sigmoid(logits).cpu().numpy()
        
        # Compute Fmax
        fmax, _ = compute_fmax(preds, Y_val)
        fmax_scores.append(fmax)
        print(f"  Model {i+1}: Fmax = {fmax:.4f}")
    
    # Convert to weights (higher Fmax = higher weight)
    # Use softmax to normalize
    fmax_array = np.array(fmax_scores)
    weights = np.exp(fmax_array * 10)  # Scale up differences
    weights = weights / weights.sum()  # Normalize
    
    print(f"\nEnsemble weights:")
    for i, (fmax, weight) in enumerate(zip(fmax_scores, weights)):
        print(f"  Model {i+1}: Fmax={fmax:.4f}, Weight={weight:.3f}")
    
    return weights.tolist()


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
n_models = 5  # Train 5 models per aspect
seeds = [42, 123, 456, 789, 2024]  # Different random seeds

# TRAIN ENSEMBLE FOR EACH ASPECT
print("\n" + "="*60)
print("TRAINING MF ENSEMBLE")
print("\n" + "="*60)

mf_models = train_ensemble_models(
    MF_MLP,
    X_tr, Y_mf_tr,
    X_val, Y_mf_val,
    aspect_name="MF",
    n_models=n_models,
    seeds=seeds,
    pos_weight_alpha=0,
    epochs=50,
    batch_size=1024,
    lr=1e-3,
    patience=10
)

In [ ]:
bp_models = train_ensemble_models(
    BP_MLP,
    X_tr, Y_bp_tr,
    X_val, Y_bp_val,
    aspect_name="BP",
    n_models=n_models,
    seeds=seeds,
    pos_weight_alpha=0,
    epochs=50,
    batch_size=1024,
    lr=1e-3,
    patience=10
)

In [ ]:
cc_models = train_ensemble_models(
    CC_MLP,
    X_tr, Y_cc_tr,
    X_val, Y_cc_val,
    aspect_name="CC",
    n_models=n_models,
    seeds=seeds,
    pos_weight_alpha=0,
    epochs=50,
    batch_size=1024,
    lr=1e-3,
    patience=10
)

In [ ]:
# ========================================
# EVALUATE ENSEMBLE ON VALIDATION
# ========================================
    
print("\n" + "="*60)
print("EVALUATING ENSEMBLE METHODS")
print("="*60)
    
# Convert validation data
Y_mf_val_np = Y_mf_val if isinstance(Y_mf_val, np.ndarray) else Y_mf_val.numpy()
Y_bp_val_np = Y_bp_val if isinstance(Y_bp_val, np.ndarray) else Y_bp_val.numpy()
Y_cc_val_np = Y_cc_val if isinstance(Y_cc_val, np.ndarray) else Y_cc_val.numpy()

# Method 1: Simple Average
print("\n--- Method 1: Simple Average ---")
    
mf_pred_avg = ensemble_predict_average(mf_models, X_val)
bp_pred_avg = ensemble_predict_average(bp_models, X_val)
cc_pred_avg = ensemble_predict_average(cc_models, X_val)
    
fmax_mf_avg, _ = compute_fmax(mf_pred_avg, Y_mf_val_np)
fmax_bp_avg, _ = compute_fmax(bp_pred_avg, Y_bp_val_np)
fmax_cc_avg, _ = compute_fmax(cc_pred_avg, Y_cc_val_np)

# Method 2: Weighted Average
print("\n--- Method 2: Weighted Average ---")
    
weights_mf = compute_ensemble_weights(mf_models, X_val, Y_mf_val_np)
weights_bp = compute_ensemble_weights(bp_models, X_val, Y_bp_val_np)
weights_cc = compute_ensemble_weights(cc_models, X_val, Y_cc_val_np)
    
mf_pred_weighted = ensemble_predict_weighted(mf_models, X_val, weights_mf)
bp_pred_weighted = ensemble_predict_weighted(bp_models, X_val, weights_bp)
cc_pred_weighted = ensemble_predict_weighted(cc_models, X_val, weights_cc)
    
fmax_mf_weighted, _ = compute_fmax(mf_pred_weighted, Y_mf_val_np)
fmax_bp_weighted, _ = compute_fmax(bp_pred_weighted, Y_bp_val_np)
fmax_cc_weighted, _ = compute_fmax(cc_pred_weighted, Y_cc_val_np)
    
print(f"MF Fmax: {fmax_mf_weighted:.4f}")
print(f"BP Fmax: {fmax_bp_weighted:.4f}")
print(f"CC Fmax: {fmax_cc_weighted:.4f}")
print(f"Average: {(fmax_mf_weighted + fmax_bp_weighted + fmax_cc_weighted)/3:.4f}")

In [ ]:
# ========================================
# SAVE BEST ENSEMBLE METHOD
# ========================================
    
avg_fmax = (fmax_mf_avg + fmax_bp_avg + fmax_cc_avg) / 3
weighted_fmax = (fmax_mf_weighted + fmax_bp_weighted + fmax_cc_weighted) / 3
    
print(f"\n{'='*60}")
print("ENSEMBLE COMPARISON")
print(f"{'='*60}")
print(f"Simple Average:  {avg_fmax:.4f}")
print(f"Weighted Average: {weighted_fmax:.4f}")
    
if weighted_fmax > avg_fmax:
    print("\n✓ Weighted average performs better! Using weighted ensemble.")
    best_method = "weighted"
    # Save weights
    ensemble_weights = {
        'MF': weights_mf,
        'BP': weights_bp,
        'CC': weights_cc
    }
else:
    print("\n✓ Simple average performs better! Using simple ensemble.")
    best_method = "average"
    ensemble_weights = None
    


In [ ]:
# Compute Fmax and thresholds
print("Computing Fmax and optimal thresholds...")
fmax_mf, thresh_mf, prec_mf, rec_mf = compute_fmax_with_details(mf_pred_avg, Y_mf_val_np)
fmax_bp, thresh_bp, prec_bp, rec_bp = compute_fmax_with_details(bp_pred_avg, Y_bp_val_np)
fmax_cc, thresh_cc, prec_cc, rec_cc = compute_fmax_with_details(cc_pred_avg, Y_cc_val_np)

# Print results
print(f"\n{'='*60}")
print("ENSEMBLE VALIDATION RESULTS")
print(f"{'='*60}")
print(f"\nMF: Fmax={fmax_mf:.4f}, Threshold={thresh_mf:.3f}, P={prec_mf:.3f}, R={rec_mf:.3f}")
print(f"BP: Fmax={fmax_bp:.4f}, Threshold={thresh_bp:.3f}, P={prec_bp:.3f}, R={rec_bp:.3f}")
print(f"CC: Fmax={fmax_cc:.4f}, Threshold={thresh_cc:.3f}, P={prec_cc:.3f}, R={rec_cc:.3f}")
print(f"\nAverage Fmax: {(fmax_mf + fmax_bp + fmax_cc)/3:.4f}")

# Save thresholds
ensemble_info = {
    'thresholds': {
        'MF': thresh_mf,
        'BP': thresh_bp,
        'CC': thresh_cc
    },
    'fmax_scores': {
        'MF': fmax_mf,
        'BP': fmax_bp,
        'CC': fmax_cc,
        'average': (fmax_mf + fmax_bp + fmax_cc) / 3
    },
    'n_models': n_models,
    'seeds': seeds
}

with open('ensemble_info.pkl', 'wb') as f:
    pickle.dump(ensemble_info, f)

print("\n✓ Saved ensemble_info.pkl")


In [ ]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm

# ============================================
# STEP 1: GENERATE ENSEMBLE TEST PREDICTIONS
# ============================================

# ============================================
# MEMORY-EFFICIENT ENSEMBLE PREDICTION
# ============================================

print("="*60)
print("GENERATING ENSEMBLE TEST PREDICTIONS (Memory-Efficient)")
print("="*60)

device = "cuda" if torch.cuda.is_available() else "cpu"
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

n_models = 5
seeds = [42, 123, 456, 789, 2024]

# --- MF Ensemble (Running Average) ---
print("\nMF ensemble predictions...")
test_pred_mf = None

for i, seed in enumerate(seeds):
    print(f"  Loading model {i+1}/{n_models} (seed={seed})...")
    
    # Load model
    model = MF_MLP(1024, len(mf_terms)).to(device)
    model.load_state_dict(torch.load(f"model_mf_seed{seed}.pt", map_location=device))
    model.eval()
    
    # Generate predictions
    with torch.no_grad():
        logits = model(X_test_tensor)
        preds = torch.sigmoid(logits).cpu().numpy()
    
    # Running average
    if test_pred_mf is None:
        test_pred_mf = preds / n_models  # First model
    else:
        test_pred_mf += preds / n_models  # Add subsequent models
    
    # Delete model to free memory
    del model
    torch.cuda.empty_cache()  # Clear GPU cache
    
    print(f"  ✓ Model {i+1} processed")

print(f"✓ MF ensemble complete: {test_pred_mf.shape}")

# --- BP Ensemble (Running Average) ---
print("\nBP ensemble predictions...")
test_pred_bp = None

for i, seed in enumerate(seeds):
    print(f"  Loading model {i+1}/{n_models} (seed={seed})...")
    
    model = BP_MLP(1024, len(bp_terms)).to(device)
    model.load_state_dict(torch.load(f"model_bp_seed{seed}.pt", map_location=device))
    model.eval()
    
    with torch.no_grad():
        logits = model(X_test_tensor)
        preds = torch.sigmoid(logits).cpu().numpy()
    
    if test_pred_bp is None:
        test_pred_bp = preds / n_models
    else:
        test_pred_bp += preds / n_models
    
    del model
    torch.cuda.empty_cache()
    
    print(f"  ✓ Model {i+1} processed")

print(f"✓ BP ensemble complete: {test_pred_bp.shape}")

# --- CC Ensemble (Running Average) ---
print("\nCC ensemble predictions...")
test_pred_cc = None

for i, seed in enumerate(seeds):
    print(f"  Loading model {i+1}/{n_models} (seed={seed})...")
    
    model = CC_MLP(1024, len(cc_terms)).to(device)
    model.load_state_dict(torch.load(f"model_cc_seed{seed}.pt", map_location=device))
    model.eval()
    
    with torch.no_grad():
        logits = model(X_test_tensor)
        preds = torch.sigmoid(logits).cpu().numpy()
    
    if test_pred_cc is None:
        test_pred_cc = preds / n_models
    else:
        test_pred_cc += preds / n_models
    
    del model
    torch.cuda.empty_cache()
    
    print(f"  ✓ Model {i+1} processed")

print(f"✓ CC ensemble complete: {test_pred_cc.shape}")

# Save ensemble predictions
np.save('ensemble_test_pred_mf.npy', test_pred_mf)
np.save('ensemble_test_pred_bp.npy', test_pred_bp)
np.save('ensemble_test_pred_cc.npy', test_pred_cc)

print(f"\n{'='*60}")
print("✓ ALL ENSEMBLE PREDICTIONS SAVED")
print(f"{'='*60}")
print(f"  ensemble_test_pred_mf.npy: {test_pred_mf.shape}")
print(f"  ensemble_test_pred_bp.npy: {test_pred_bp.shape}")
print(f"  ensemble_test_pred_cc.npy: {test_pred_cc.shape}")

In [ ]:
import numpy as np
from goatools.obo_parser import GODag
go_dag = GODag("/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo")


In [ ]:
# ============================================
# STEP 2: LOAD OPTIMAL THRESHOLDS
# ============================================

print("\n" + "="*60)
print("LOADING OPTIMAL THRESHOLDS")
print("="*60)

# These were computed from validation set during ensemble training
# You should have saved them like this:
# with open('ensemble_thresholds.pkl', 'wb') as f:
#     pickle.dump({'MF': thresh_mf, 'BP': thresh_bp, 'CC': thresh_cc}, f)

with open('/kaggle/working/ensemble_info.pkl', 'rb') as f:
    ensemble_info = pickle.load(f)
    optimal_thresholds = ensemble_info["thresholds"]

thresh_mf = optimal_thresholds['MF']
thresh_bp = optimal_thresholds['BP']
thresh_cc = optimal_thresholds['CC']

print(f"\nOptimal thresholds:")
print(f"  MF: {thresh_mf:.3f}")
print(f"  BP: {thresh_bp:.3f}")
print(f"  CC: {thresh_cc:.3f}")

# ============================================
# STEP 3: CREATE SUBMISSION WITH THRESHOLDS + 1500 LIMIT
# ============================================

def create_final_submission(
    pred_mf, pred_bp, pred_cc,
    thresh_mf, thresh_bp, thresh_cc,
    mf_terms, bp_terms, cc_terms,
    protein_ids, go_dag,
    max_terms=1500,
    min_predictions=5
):
    """
    Create CAFA submission with:
    1. Thresholding
    2. Hierarchical propagation
    3. 1500-term limit per protein
    4. Fallback for proteins with no predictions
    """
    print(f"\n{'='*60}")
    print("CREATING FINAL SUBMISSION")
    print(f"{'='*60}")
    print(f"\nProcessing {len(protein_ids):,} proteins...")
    
    submissions = []
    proteins_with_fallback = 0
    
    for i in tqdm(range(len(protein_ids)), desc="Creating submission"):
        protein_id = protein_ids[i]
        
        # ==========================================
        # STEP 3.1: Apply thresholds
        # ==========================================
        protein_predictions = {}
        
        # MF terms above threshold
        for j, go_id in enumerate(mf_terms):
            if pred_mf[i, j] >= thresh_mf:
                protein_predictions[go_id] = float(pred_mf[i, j])
        
        # BP terms above threshold
        for j, go_id in enumerate(bp_terms):
            if pred_bp[i, j] >= thresh_bp:
                protein_predictions[go_id] = float(pred_bp[i, j])
        
        # CC terms above threshold
        for j, go_id in enumerate(cc_terms):
            if pred_cc[i, j] >= thresh_cc:
                protein_predictions[go_id] = float(pred_cc[i, j])
        
        # ==========================================
        # FALLBACK: If no predictions, use top-N
        # ==========================================
        if len(protein_predictions) == 0:
            proteins_with_fallback += 1
            
            # Collect all predictions
            all_scores = []
            for j, go_id in enumerate(mf_terms):
                all_scores.append((go_id, float(pred_mf[i, j])))
            for j, go_id in enumerate(bp_terms):
                all_scores.append((go_id, float(pred_bp[i, j])))
            for j, go_id in enumerate(cc_terms):
                all_scores.append((go_id, float(pred_cc[i, j])))
            
            # Take top min_predictions
            all_scores.sort(key=lambda x: x[1], reverse=True)
            for go_id, score in all_scores[:min_predictions]:
                protein_predictions[go_id] = score
        
        # ==========================================
        # STEP 3.2: Hierarchical propagation
        # ==========================================
        propagated = {}
        
        for go_id, score in protein_predictions.items():
            # Add term itself
            propagated[go_id] = score
            
            # Add all parents
            if go_id in go_dag:
                try:
                    parents = go_dag[go_id].get_all_parents()
                    for parent in parents:
                        propagated[parent] = max(
                            propagated.get(parent, 0),
                            score * 0.95  # Slight decay
                        )
                except:
                    pass
        
        # ==========================================
        # STEP 3.3: Apply 1500-term limit
        # ==========================================
        if len(propagated) > max_terms:
            # Sort by score, keep top max_terms
            sorted_terms = sorted(propagated.items(), key=lambda x: x[1], reverse=True)
            selected = dict(sorted_terms[:max_terms])
            
            # Ensure hierarchical consistency
            must_include = set(selected.keys())
            for go_id in list(selected.keys()):
                if go_id in go_dag:
                    try:
                        parents = go_dag[go_id].get_all_parents()
                        for parent in parents:
                            if parent not in must_include and parent in propagated:
                                must_include.add(parent)
                                selected[parent] = propagated[parent]
                    except:
                        pass
            
            # Remove lowest-scoring leaves if over limit
            while len(selected) > max_terms:
                leaves = []
                for go_id in selected.keys():
                    is_leaf = True
                    if go_id in go_dag:
                        try:
                            children = go_dag[go_id].children
                            for child in children:
                                if child.id in selected:
                                    is_leaf = False
                                    break
                        except:
                            pass
                    if is_leaf:
                        leaves.append(go_id)
                
                if not leaves:
                    lowest = min(selected, key=selected.get)
                    del selected[lowest]
                else:
                    lowest_leaf = min(leaves, key=lambda x: selected[x])
                    del selected[lowest_leaf]
            
            propagated = selected
        
        # ==========================================
        # STEP 3.4: Add to submission
        # ==========================================
        for go_id, score in propagated.items():
            submissions.append({
                'protein_id': protein_id,
                'go_id': go_id,
                'score': score
            })
    
    if proteins_with_fallback > 0:
        print(f"\n⚠️  {proteins_with_fallback} proteins used fallback predictions")
    
    return submissions


# ============================================
# GENERATE SUBMISSION
# ============================================

submissions = create_final_submission(
    test_pred_mf, test_pred_bp, test_pred_cc,
    thresh_mf, thresh_bp, thresh_cc,
    mf_terms, bp_terms, cc_terms,
    df_test['protein_id'].tolist(),
    go_dag,
    max_terms=1500,
    min_predictions=5
)

# ============================================
# CREATE SUBMISSION DATAFRAME
# ============================================

submission_df = pd.DataFrame(submissions)

print(f"\n{'='*60}")
print("SUBMISSION STATISTICS")
print(f"{'='*60}")
print(f"Total predictions: {len(submission_df):,}")
print(f"Unique proteins: {submission_df['protein_id'].nunique():,}")
print(f"Unique GO terms: {submission_df['go_id'].nunique():,}")

# Verify 1500 limit
terms_per_protein = submission_df.groupby('protein_id').size()
print(f"\nTerms per protein:")
print(f"  Mean: {terms_per_protein.mean():.1f}")
print(f"  Median: {terms_per_protein.median():.1f}")
print(f"  Min: {terms_per_protein.min()}")
print(f"  Max: {terms_per_protein.max()}")

assert terms_per_protein.max() <= 1500, "ERROR: Some proteins exceed 1500 terms!"
print(f"\n✓ All proteins within 1500 term limit")

# Check score distribution
print(f"\nScore distribution:")
print(f"  Mean: {submission_df['score'].mean():.4f}")
print(f"  Median: {submission_df['score'].median():.4f}")
print(f"  Min: {submission_df['score'].min():.4f}")
print(f"  Max: {submission_df['score'].max():.4f}")

# ============================================
# SAVE SUBMISSION FILE
# ============================================

# CAFA format: protein_id \t go_id \t score (NO HEADER)
submission_df[['protein_id', 'go_id', 'score']].to_csv(
    'submission.tsv',
    sep='\t',
    index=False,
    header=False
)

print(f"\n{'='*60}")
print("✓ SUBMISSION SAVED: submission.tsv")
print(f"{'='*60}")

# ============================================
# SANITY CHECKS
# ============================================

print(f"\nRunning sanity checks...")

# Check 1: Coverage
test_proteins = set(df_test['protein_id'].tolist())
submission_proteins = set(submission_df['protein_id'].unique())
missing = test_proteins - submission_proteins

if len(missing) > 0:
    print(f"⚠️  {len(missing)} proteins missing from submission")
else:
    print(f"✓ All {len(test_proteins)} test proteins have predictions")

# Check 2: Valid GO terms
invalid_terms = [t for t in submission_df['go_id'].unique() if t not in go_dag]
if len(invalid_terms) > 0:
    print(f"⚠️  {len(invalid_terms)} invalid GO terms")
else:
    print(f"✓ All GO terms are valid")

# Check 3: Score range
if (submission_df['score'] < 0).any() or (submission_df['score'] > 1).any():
    print(f"⚠️  Some scores outside [0, 1] range")
else:
    print(f"✓ All scores in valid range")

print(f"\n{'='*60}")
print("ENSEMBLE SUBMISSION COMPLETE!")
print(f"{'='*60}")